# Helmet Detection — YOLOv8 Training Notebook
AIRI Team PITB — AI Internship Task 1

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Ultralytics

In [ ]:
!pip install -U ultralytics

## 3. Check GPU

In [ ]:
import torch
print('GPU Available:', torch.cuda.is_available())
!nvidia-smi

## 4. Sanity check dataset structure

In [ ]:
import os
base = '/content/drive/MyDrive/cv_project/dataset'
for split in ['train', 'val', 'test']:
    img_dir = f'{base}/images/{split}'
    lbl_dir = f'{base}/labels/{split}'
    n_img = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbl = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f'{split}: {n_img} images, {n_lbl} labels')

## 5. Load baseline model

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')

## 6. Train

In [ ]:
results = model.train(
    data='/content/drive/MyDrive/cv_project/dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=8,
    pretrained=True,
    project='/content/drive/MyDrive/cv_project/outputs/training_results',
    name='yolov8n_baseline'
)

## 7. Evaluate (val + test)

In [ ]:
metrics = model.val(split='val')
print(metrics.box.map, metrics.box.map50, metrics.box.mp, metrics.box.mr)

In [ ]:
metrics_test = model.val(split='test')
print(metrics_test.box.map, metrics_test.box.map50, metrics_test.box.mp, metrics_test.box.mr)

## 8. Inference on unseen test images

In [ ]:
best_model = YOLO('/content/drive/MyDrive/cv_project/outputs/training_results/yolov8n_baseline/weights/best.pt')

results = best_model.predict(
    source='/content/drive/MyDrive/cv_project/dataset/images/test',
    conf=0.35,
    save=True,
    project='/content/drive/MyDrive/cv_project/outputs/predictions',
    name='test_predictions'
)

## 9. Copy best.pt to models/

In [ ]:
import shutil
shutil.copy(
    '/content/drive/MyDrive/cv_project/outputs/training_results/yolov8n_baseline/weights/best.pt',
    '/content/drive/MyDrive/cv_project/models/best.pt'
)